In [15]:
import numpy as np

data = np.array([[1, 2], [3, 4], [5, 6]])
print(data[:, 0:1])

[[1]
 [3]
 [5]]


In [18]:
mean = np.mean(data, axis = 0)
mean

array([3., 4.])

In [19]:
centered = data - mean
centered

array([[-2., -2.],
       [ 0.,  0.],
       [ 2.,  2.]])

In [ ]:
import numpy as np

# --- Step 1: Data Initialization (fixed as per constraint) ---
# Data (flexible size) - np.array is allowed here
data = np.array([[2, 1], [4, 3], [5, 4]], dtype=float)
n_rows = len(data)  # Number of data points (samples)
n_cols = len(data[0]) # Number of features/columns (for clarity)

print("Original data:\n", data)

# --- Step 2: Center the data (Mean Calculation and Subtraction) ---

# Calculate mean for each feature (column)
mean_features = [0.0] * n_cols # Use 0.0 for float
for j in range(n_cols): # Iterate through each column
    column_sum = 0.0 # Accumulator for current column's sum
    for i in range(n_rows): # Iterate through each row in the current column
        column_sum += data[i][j]
    mean_features[j] = column_sum / n_rows # Store the calculated mean

print("\nCalculated means for each feature:", mean_features)

# Center the data by subtracting the mean of each feature
centered_data = [] # Initialize as empty list
for i in range(n_rows): # Iterate through rows
    current_row = [] # Build each row of centered data
    for j in range(n_cols): # Iterate through columns (features)
        current_row.append(data[i][j] - mean_features[j])
    centered_data.append(current_row) # Add completed row to centered_data

print("\nCentered data:\n", centered_data)

# --- Step 3: Covariance Matrix Calculation ---

# Initialize covariance matrix (n_cols x n_cols)
covariance_matrix = []
for _ in range(n_cols):
    covariance_matrix.append([0.0] * n_cols)

# Calculate covariance between each pair of features
# cov[i][j] is the covariance between feature 'i' and feature 'j'
for i in range(n_cols): # First feature index
    for j in range(n_cols): # Second feature index
        sum_of_products = 0.0 # Accumulator for sum of (centered_i * centered_j)
        for k in range(n_rows): # Iterate through observations (rows)
            # Multiply centered value of feature 'i' by centered value of feature 'j' for current observation 'k'
            sum_of_products += centered_data[k][i] * centered_data[k][j]

        # Divide by (n_rows - 1) for sample covariance
        if n_rows > 1: # Avoid division by zero for single data point
            covariance_matrix[i][j] = sum_of_products / (n_rows - 1)
        else:
            covariance_matrix[i][j] = 0.0 # Covariance not meaningful for single point

print("\nCovariance matrix:\n", covariance_matrix)

# --- Step 4: Eigenvalues & Eigenvectors (for 2x2 matrix only) ---
# This part is highly specific to 2x2 matrices and uses the characteristic equation.
# Generalizing this manually for larger matrices is extremely complex (requires finding roots of higher-order polynomials).

# Extract elements for a 2x2 covariance matrix [ [a, b], [c, d] ]
# Ensure n_cols is 2 for this part to be valid
if n_cols != 2:
    print("\nWarning: Eigenvalue/Eigenvector calculation is for 2x2 matrix only. This code needs generalization for >2 features.")
    exit() # Exit or handle error if not 2 features

a, b = covariance_matrix[0][0], covariance_matrix[0][1]
c, d = covariance_matrix[1][0], covariance_matrix[1][1]

# Characteristic equation: lambda^2 - (a+d)lambda + (ad-bc) = 0
coeff_a_quad = 1.0
coeff_b_quad = -(a + d)
coeff_c_quad = a * d - b * c

# Discriminant for quadratic formula: disc = b^2 - 4ac
discriminant = coeff_b_quad**2 - 4 * coeff_a_quad * coeff_c_quad

if discriminant < 0:
    print("\nError: Complex eigenvalues (this method assumes real eigenvalues).")
    # For simplicity in this manual implementation, we'll assume real eigenvalues.
    # In practice, floating point inaccuracies can sometimes lead to tiny negative discriminant.
    # A robust solution would handle complex numbers or check for near-zero discriminant.
    discriminant = 0 # Force to 0 if very small negative due to float precision

# Calculate eigenvalues using quadratic formula
lambda1 = (-coeff_b_quad + discriminant**0.5) / (2 * coeff_a_quad)
lambda2 = (-coeff_b_quad - discriminant**0.5) / (2 * coeff_a_quad)
eigenvalues = [lambda1, lambda2]
print("\nEigenvalues:", eigenvalues)

# Pick largest eigenvalue
if lambda1 >= lambda2: # Use >= to consistently pick lambda1 if they are equal
    max_lambda = lambda1
    other_lambda = lambda2
else:
    max_lambda = lambda2
    other_lambda = lambda1
print("Largest eigenvalue:", max_lambda)

# Calculate Eigenvector for largest eigenvalue (specific for 2x2)
# For a 2x2 matrix A, (A - lambda*I)v = 0
# The equations are:
# (a - lambda)v1 + b*v2 = 0
# c*v1 + (d - lambda)v2 = 0
# From the first equation (assuming b != 0): v2 = -(a - lambda)/b * v1
# Let v1 = 1, then v2 = -(a - lambda)/b
# Handle cases where b (cov[0][1]) is zero for principal component direction
eigenvector_largest_lambda = [0.0] * n_cols

# Use the first row of (Cov - lambda*I) to find relationship
# (cov[0][0] - max_lambda) * v[0] + cov[0][1] * v[1] = 0
# If cov[0][1] is non-zero, we can solve for v[1] in terms of v[0]
if abs(covariance_matrix[0][1]) > 1e-9: # Check if 'b' is effectively non-zero
    eigenvector_largest_lambda[0] = 1.0 # Arbitrarily set v[0] to 1
    eigenvector_largest_lambda[1] = -(covariance_matrix[0][0] - max_lambda) / covariance_matrix[0][1] * eigenvector_largest_lambda[0]
elif abs(covariance_matrix[1][0]) > 1e-9: # If 'b' is zero, check 'c' (cov[1][0])
    eigenvector_largest_lambda[1] = 1.0 # Arbitrarily set v[1] to 1
    eigenvector_largest_lambda[0] = -(covariance_matrix[1][1] - max_lambda) / covariance_matrix[1][0] * eigenvector_largest_lambda[1]
else: # If both off-diagonals are zero (diagonal covariance matrix)
    # Eigenvectors are [1, 0] and [0, 1] if eigenvalues are distinct
    # Or any vector if eigenvalues are the same (e.g., identity matrix)
    if abs(covariance_matrix[0][0] - max_lambda) < 1e-9: # If (a - lambda) is zero
        eigenvector_largest_lambda = [1.0, 0.0] # [1, 0] is eigenvector for this case
    elif abs(covariance_matrix[1][1] - max_lambda) < 1e-9: # If (d - lambda) is zero
        eigenvector_largest_lambda = [0.0, 1.0] # [0, 1] is eigenvector for this case
    else:
        # This case should ideally not happen if max_lambda is a true eigenvalue
        print("Warning: Could not determine eigenvector for diagonal matrix.")
        eigenvector_largest_lambda = [0.0, 0.0] # Fallback


# Normalize the eigenvector (using np.array for magnitude calculation as allowed)
# Magnitude = sqrt(v1^2 + v2^2)
magnitude = (eigenvector_largest_lambda[0]**2 + eigenvector_largest_lambda[1]**2)**0.5
if magnitude > 1e-9: # Avoid division by zero
    eigenvector_largest_lambda[0] /= magnitude
    eigenvector_largest_lambda[1] /= magnitude
else:
    print("Warning: Eigenvector has zero magnitude. Cannot normalize.")

print("Eigenvector for largest λ (normalized):", eigenvector_largest_lambda)

# --- Step 5: Project Data onto Principal Component ---

# Projected data will be 1D (n_rows x 1)
projected_data = [0.0] * n_rows

for i in range(n_rows): # Iterate through each data point (row)
    current_projection = 0.0 # Accumulator for dot product
    for k in range(n_cols): # Iterate through each feature
        # Dot product of centered_data[i] (row vector) and eigenvector (column vector)
        current_projection += centered_data[i][k] * eigenvector_largest_lambda[k]
    projected_data[i] = current_projection

print("\nProjected data (1D):\n", projected_data)

